<a href="https://colab.research.google.com/github/JariusF10-8/bt4222/blob/main/notebooks/features/feature_engineering_chris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Category encoding, historical CTR and entity prominence

This notebook builds three groups of features from the preprocessed data:

| Part | Feature group | Output |
|---|---|---|
| A | Category encoding | Codes for category and (category, subcategory) fitted on training articles, plus a history subcategory match feature |
| B | Historical popularity / CTR | Article, subcategory, user and user-article click counts and rates, using only interactions before each impression |
| C | Entity prominence (insight 4) | Title-start entity flags, first entity positions and position-weighted entity vectors |

The functions are in `src/category_encoding.py`, `src/history_features.py`, `src/entity_prominence.py` and `src/feature_diagnostics.py`, with tests in `tests/`. Validation labels are only used to check the features. Test labels are not used.

Run from top to bottom. Colab needs about 6 GB RAM and 15 to 20 minutes.

## 1. Setup

In Colab the cell asks for the repository ZIP if the code is not there yet, as in `feature_engineering_zp.ipynb`. Locally it finds the repository folder.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules


def find_project_root():
    configured = os.environ.get('BT4222_PROJECT_ROOT')
    options = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    for folder in options:
        if (folder / 'src/data_loading.py').is_file():
            return folder.resolve()
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None and IN_COLAB:
    import io
    import tempfile
    import zipfile
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
    target = Path(tempfile.mkdtemp(prefix='bt4222_', dir='/content'))
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(target)
    PROJECT_ROOT = next(target.rglob('src/data_loading.py')).parent.parent
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(PROJECT_ROOT / 'requirements.txt')], check=True)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook inside the repository or set BT4222_PROJECT_ROOT.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ['BT4222_PROJECT_ROOT'] = str(PROJECT_ROOT)
print('Repository found:', PROJECT_ROOT.name)

Repository found: bt4222-main


In [2]:
import gc
import hashlib
import json
import time

import numpy as np
import pandas as pd
from IPython.display import display

from src.preprocessing import build_dataset
from src.category_encoding import (RARE, UNKNOWN, candidate_history_subcategory_match,
                                   encode_categories, fit_category_vocab, subcategory_key)
from src.history_features import (FEATURE_COLUMNS as HISTORY_COLUMNS, SMOOTHING, WINDOW_HOURS,
                                  build_exposure_log, compute_history_features, mix_by_impression)
from src.entity_prominence import (aggregate_entity_vectors, article_prominence_features,
                                   entity_mentions, load_entity_vectors, mention_weights)
from src.feature_diagnostics import impression_metrics, paired_difference, univariate_table

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SHARED = Path('/content/drive/MyDrive/BT4222 Group Project')
    zipeng_v2 = SHARED / 'feature_engineering_zp/data/processed/v2'
    default_processed = zipeng_v2 if (zipeng_v2 / 'manifest.json').is_file() else SHARED / 'feature_engineering_chris/data/processed/v1'
    os.environ.setdefault('BT4222_PROCESSED_DIR', str(default_processed))
    os.environ.setdefault('BT4222_CHRIS_FEATURES_DIR', str(SHARED / 'feature_engineering_chris/outputs/features/chris_v1'))

RAW_DIR = Path(os.environ.get('MIND_RAW_DIR', PROJECT_ROOT / 'data/raw'))
PROCESSED_DIR = Path(os.environ.get('BT4222_PROCESSED_DIR', PROJECT_ROOT / 'data/processed/v1'))
OUTPUT_DIR = Path(os.environ.get('BT4222_CHRIS_FEATURES_DIR', PROJECT_ROOT / 'outputs/features/chris_v1'))

if not (PROCESSED_DIR / 'manifest.json').is_file():
    display(build_dataset(RAW_DIR, PROCESSED_DIR, require_test=True))
manifest = json.loads((PROCESSED_DIR / 'manifest.json').read_text())
PARTITIONS = manifest['partitions']
if PARTITIONS != ['train', 'validation', 'test']:
    raise ValueError(f'Expected train, validation and test partitions; found {PARTITIONS}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for partition in PARTITIONS:
    (OUTPUT_DIR / partition).mkdir(exist_ok=True)
print('Processed dataset version:', manifest['dataset_version'], '| partitions:', ', '.join(PARTITIONS))
display(pd.read_csv(PROCESSED_DIR / 'split_summary.csv')[['partition', 'earliest_time', 'latest_time', 'events', 'candidate_exposures', 'clicked_candidates']])

Processed dataset version: v1 | partitions: train, validation, test


,partition,earliest_time,latest_time,events,candidate_exposures,clicked_candidates
0,train,2019-11-09 00:00:19,2019-11-12 23:59:25,95071,3389342,141558
1,validation,2019-11-13 00:01:00,2019-11-14 23:59:13,61894,2454102,94786
2,test,2019-11-15 00:00:01,2019-11-15 23:58:03,73152,2740998,111383


## 2. Loaders

These loaders read only the columns each feature needs and store repeated IDs as categories. This keeps the 8.6 million candidate rows under about 3 GB.

In [3]:
def read_news(partition, columns):
    news = pd.read_csv(PROCESSED_DIR / partition / 'news.csv', usecols=columns, dtype='string', keep_default_na=False)
    if 'metadata_missing' in news:
        news['metadata_missing'] = news['metadata_missing'].eq('True')
    for column in ('title_entities_parsed', 'abstract_entities_parsed'):
        if column in news:
            news[column] = news[column].map(json.loads)
    return news


def read_events(partition, history=False):
    columns = ['event_key', 'user_id', 'timestamp'] + (['history_ids'] if history else [])
    events = pd.read_csv(PROCESSED_DIR / partition / 'events.csv', usecols=columns, keep_default_na=False,
                         dtype={'event_key': 'string', 'user_id': 'string'}, parse_dates=['timestamp'])
    if history:
        events['history_ids'] = events['history_ids'].map(json.loads)
    return events


def read_candidates(partition, label=True):
    columns = ['event_key', 'candidate_position', 'news_id'] + (['label'] if label else [])
    return pd.read_csv(PROCESSED_DIR / partition / 'candidates.csv', usecols=columns, keep_default_na=False,
                       dtype={'event_key': 'category', 'news_id': 'category', 'candidate_position': 'int32', 'label': 'int8'})


def write_candidate_table(frame, partition, name):
    path = OUTPUT_DIR / partition / f'{name}.csv.gz'
    frame.to_csv(path, index=False, float_format='%.6g', compression={'method': 'gzip', 'compresslevel': 5})
    return path


CATEGORY_COLUMNS = ['news_id', 'category', 'subcategory', 'metadata_missing']
diagnostics = {}
started = time.time()

## Part A. Category encoding

The interim report found 17 categories and 264 subcategory names. 13 subcategory names appear under more than one category (for example `video`, `news` and `recipes`), and many subcategories are rare. Every validation category appears in training, but a few subcategories do not.

1. `category_id`: training categories sorted and numbered from 1, with 0 for unknown. This is the same rule as `feature_engineering_zp.ipynb`.
2. `subcategory_id`: the (category, subcategory) pair, so `news/video` and `video/video` stay different. Pairs with fewer than `MIN_ARTICLES = 5` training articles share one `__RARE__` code per category. A pair never seen in training also gets its category's rare code, and an unseen category gets 0. The value 5 follows the interim report's rare subcategory count and is not tuned.
3. The codes are fitted on training articles with metadata and applied unchanged to validation and test.

The codes are for embedding layers in the neural models. One-hot category columns are already in Zipeng's notebook, so they are not repeated here.

In [4]:
MIN_ARTICLES = 5
train_news_categories = read_news('train', CATEGORY_COLUMNS)
vocab = fit_category_vocab(train_news_categories, min_articles=MIN_ARTICLES)
pairs = vocab['pairs']
print(f"Categories (excluding unknown): {len(vocab['category']) - 1}")
print(f"Training (category, subcategory) pairs: {len(pairs)}; rare pairs (< {MIN_ARTICLES} articles): {int(pairs['bucket'].eq(RARE).sum())}, "
      f"covering {int(pairs.loc[pairs['bucket'].eq(RARE), 'train_articles'].sum())} articles")
print(f"Subcategory codes after merging (including 0 = unknown): {len(vocab['subcategory'])}")
names = pairs.groupby('subcategory')['category'].nunique()
print('Subcategory names under more than one category:', ', '.join(sorted(names[names > 1].index)))
display(vocab['category'])
display(vocab['subcategory'].head(12))

Categories (excluding unknown): 17
Training (category, subcategory) pairs: 276; rare pairs (< 5 articles): 107, covering 192 articles
Subcategory codes after merging (including 0 = unknown): 187
Subcategory names under more than one category: animals, causes, holidays, humor, news, newstrends, personalfinance, pregnancyparenting, recipes, science, topnews, video, voices


,category_id,category,train_articles
0,0,__UNKNOWN__,0
1,1,autos,1425
2,2,entertainment,529
3,3,finance,2585
4,4,foodanddrink,2166
5,5,health,1673
6,6,kids,14
7,7,lifestyle,2092
8,8,middleeast,2
9,9,movies,561


,subcategory_id,category,bucket,train_articles,merged_subcategories
0,0,__UNKNOWN__,__UNKNOWN__,0,0
1,1,autos,__RARE__,13,7
2,2,autos,autosbuying,28,1
3,3,autos,autosclassics,108,1
4,4,autos,autosenthusiasts,216,1
5,5,autos,autoshybrids,6,1
6,6,autos,autosluxury,19,1
7,7,autos,autosmotorcycles,102,1
8,8,autos,autosnews,693,1
9,9,autos,autosownership,25,1


In [5]:
encoded_by_partition, coverage_rows = {}, []
for partition in PARTITIONS:
    news = train_news_categories if partition == 'train' else read_news(partition, CATEGORY_COLUMNS)
    encoded = encode_categories(news, vocab)
    encoded_by_partition[partition] = encoded
    exposures = read_candidates(partition, label=False)[['news_id']].astype(str).merge(
        encoded, on='news_id', how='left', validate='many_to_one')
    unseen_pairs = set(zip(news['category'], news['subcategory'])) - set(zip(pairs['category'], pairs['subcategory']))
    unseen_categories = sorted(set(news.loc[encoded['category_unknown'].eq(1).to_numpy(), 'category']) - {UNKNOWN})
    coverage_rows.append({
        'partition': partition, 'articles': len(encoded),
        'unknown_category_articles': int(encoded['category_unknown'].sum()),
        'unseen_categories': ', '.join(unseen_categories) or '-',
        'pairs_not_in_training': len(unseen_pairs),
        'backoff_articles': int(encoded['subcategory_backoff'].sum()),
        'exposures_unknown_category_%': round(100 * exposures['category_unknown'].mean(), 3),
        'exposures_subcategory_backoff_%': round(100 * exposures['subcategory_backoff'].mean(), 3),
    })
category_coverage = pd.DataFrame(coverage_rows)
display(category_coverage)

,partition,articles,unknown_category_articles,unseen_categories,pairs_not_in_training,backoff_articles,exposures_unknown_category_%,exposures_subcategory_backoff_%
0,train,43370,0,-,0,192,0.0,1.552
1,validation,38421,0,-,7,190,0.0,1.174
2,test,42416,1,games,14,212,0.0,1.867


### A2. History subcategory match

`candidate_history_subcategory_share` is the share of the user's history clicks (known subcategories only, repeats kept) that are in the candidate's (category, subcategory) pair. It needs no labels. An empty history gives 0 with `history_subcategory_missing = 1`. It adds the subcategory level to Zipeng's `candidate_history_category_share`.

In [6]:
subcategory_keys = {}
for partition in PARTITIONS:
    news = train_news_categories if partition == 'train' else read_news(partition, CATEGORY_COLUMNS)
    subcategory_keys[partition] = pd.Series(subcategory_key(news).to_numpy(), index=news['news_id'].astype(str).to_numpy())
    events = read_events(partition, history=True)
    candidates = read_candidates(partition)
    match = candidate_history_subcategory_match(events, candidates, news)
    columns = ['event_key', 'candidate_position', 'candidate_history_subcategory_share',
               'candidate_history_subcategory_count', 'history_subcategory_missing']
    write_candidate_table(match[columns], partition, 'candidate_subcategory_match')
    if partition == 'validation':
        check = match.assign(label=candidates['label'].to_numpy())
        diagnostics['subcategory_match_validation'] = univariate_table(check, ['candidate_history_subcategory_share'])
    print(f'{partition}: {len(match):,} candidate rows; mean share {match["candidate_history_subcategory_share"].mean():.4f}; '
          f'empty-history rows {int(match["history_subcategory_missing"].sum()):,}')
    del events, candidates, match
    _ = gc.collect()
display(diagnostics['subcategory_match_validation'])

train: 3,389,342 candidate rows; mean share 0.0408; empty-history rows 61,834


validation: 2,454,102 candidate rows; mean share 0.0357; empty-history rows 58,608


test: 2,740,998 candidate rows; mean share 0.0324; empty-history rows 82,907


,feature,direction,impressions,AUC,MRR,nDCG@5,nDCG@10
0,candidate_history_subcategory_share,higher is better,61894,0.5885,0.2729,0.2894,0.3478


Ranking each validation impression by this feature alone gives AUC 0.589 and nDCG@10 0.348, against 0.500 and about 0.270 for a random ranking.

## Part B. Historical popularity and CTR

A feature for an impression at time *t* only counts exposures and clicks before a cutoff, and the cutoff is never later than *t*. The impression itself and anything after it are never used.

| Mode | Training rows | Validation rows | Test rows |
|---|---|---|---|
| `online` | before *t* | before *t* (includes earlier validation impressions) | before *t* (includes earlier test impressions) |
| `nightly` | before midnight of that day | before midnight of that day | before 15 Nov midnight, same as `frozen` |
| `frozen` | before *t* | before 13 Nov midnight | before 15 Nov midnight |

- `online` is a platform that logs clicks as they happen.
- `nightly` is a platform that recomputes counts once a night, so training rows are as out of date as test rows. The test set is a single day, so its nightly features are the same as its frozen features.
- `frozen` follows the interim report. Training rows are up to date, but validation and test only see counts from before their period starts.

The 1, 6 and 24 hour windows end at the cutoff.

**What the group uses.** The shared feature bundle (`03_Unified_Feature_Engineering.ipynb`) has its own `train_only` mode. Training rows use earlier training clicks, and validation and test rows only use training-period clicks, so validation and test labels are never inputs. For validation this is the same cutoff as `frozen`. Training rows are still up to date, which is the mismatch tested in B4. `online` validation features are kept here only to measure how much real-time logging would be worth.

| Column | Meaning |
|---|---|
| `art_prior_exposures`, `art_prior_clicks` | Times the article was shown and clicked before the cutoff |
| `art_exposures_{1,6,24}h`, `art_clicks_{1,6,24}h` | The same within the last 1, 6 and 24 hours |
| `art_prior_ctr`, `art_ctr_24h` | Smoothed article click rate, pulled towards its subcategory rate |
| `subcat_prior_ctr` | Smoothed click rate of the (category, subcategory) pair, pulled towards the training click rate |
| `art_hours_since_first_exposure`, `art_seen_before` | Hours since the article was first shown (MIND has no publish time) and whether it was shown before |
| `user_prior_events`, `user_prior_exposures`, `user_prior_clicks`, `user_prior_ctr` | The user's own earlier activity |
| `user_article_prior_exposures`, `user_article_prior_clicks`, `user_article_skipped_before` | Whether this user was already shown this article and did not click it |

Smoothed rates use `(clicks + m x prior) / (exposures + m)`, with m = 100 for subcategories and 20 for articles and users. The prior is the training click rate. This lets a new article start at its topic's rate instead of 0. The m values are not tuned.

**Difference from the previous project.** Group 10 computed CTR on the negative-sampled table, so every article was close to 1/5, each row's own click was inside its CTR, and every unseen test article got the same value. Here every rate uses the full candidate log and only earlier rows.

In [7]:
log_inputs = {partition: (read_events(partition), read_candidates(partition)) for partition in PARTITIONS}
log = build_exposure_log(log_inputs, subcategory_keys)
candidate_keys = {partition: candidates[['event_key', 'candidate_position']].copy() for partition, (_, candidates) in log_inputs.items()}
del log_inputs
_ = gc.collect()
print(f'Exposure log: {len(log):,} candidate rows, {log["event"].nunique():,} impressions, '
      f'{log["article"].nunique():,} articles, {log["user"].nunique():,} users')
print('Training click rate (global prior):', round(float(log.loc[log['partition'].eq(0), 'label'].mean()), 5))

Exposure log: 8,584,442 candidate rows, 230,117 impressions, 22,771 articles, 94,057 users
Training click rate (global prior): 0.04177


In [8]:
history_frames = {}
for mode in ('online', 'nightly', 'frozen'):
    tick = time.time()
    history_frames[mode] = compute_history_features(log, mode=mode)
    print(f'{mode}: computed {len(HISTORY_COLUMNS)} features in {time.time() - tick:.0f}s')

online: computed 20 features in 20s


nightly: computed 20 features in 18s


frozen: computed 20 features in 20s


### B2. Leakage checks on the real data

1. **Brute force.** For random rows in every partition and mode, recount the history directly from the log and compare with the fast code.
2. **Label flip.** Flip every label at or after a cut time inside the validation period and recompute. Features of rows at or before the cut must not change.

In [9]:
origin = pd.Timestamp(log.attrs['origin'])


def brute_force(row, mode):
    t = log.at[row, 't']
    upper = t
    if mode == 'frozen' and log.at[row, 'partition'] != 0:
        first = log.loc[log['partition'].eq(log.at[row, 'partition']), 't'].min()
        upper = min(t, max(0, int(((origin + pd.Timedelta(seconds=int(first))).normalize() - origin) / pd.Timedelta(seconds=1))))
    if mode == 'nightly':
        midnight = (origin + pd.Timedelta(seconds=int(t))).normalize()
        upper = max(0, int((midnight - origin) / pd.Timedelta(seconds=1)))
    past = log.loc[log['t'].lt(upper)]
    same = past.loc[past['article'].eq(log.at[row, 'article'])]
    day = same.loc[same['t'].ge(upper - 24 * 3600)]
    pair = same.loc[same['user'].eq(log.at[row, 'user'])]
    return {'art_prior_exposures': len(same), 'art_prior_clicks': int(same['label'].sum()),
            'art_exposures_24h': len(day), 'art_clicks_24h': int(day['label'].sum()),
            'user_article_prior_exposures': len(pair), 'user_article_prior_clicks': int(pair['label'].sum())}


rng = np.random.default_rng(42)
mismatches, checked = 0, 0
for mode in ('online', 'nightly', 'frozen'):
    for code_ in (0, 1, 2):
        rows = rng.choice(np.flatnonzero(log['partition'].to_numpy() == code_), size=25, replace=False)
        for row in rows:
            expected = brute_force(row, mode)
            got = history_frames[mode].loc[row, list(expected)]
            mismatches += int((got.to_numpy() != np.array(list(expected.values()))).any())
            checked += 1
print(f'Brute-force check: {checked} rows, {mismatches} mismatches')
assert mismatches == 0

validation_t = log.loc[log['partition'].eq(1), 't']
cut = int(validation_t.median())
perturbed = log.copy()
later = perturbed['t'].ge(cut).to_numpy()
perturbed.loc[later, 'label'] = 1 - perturbed.loc[later, 'label']
p0 = float(log.loc[log['partition'].eq(0), 'label'].mean())
reference = history_frames['online']
changed = compute_history_features(perturbed, mode='online', prior_ctr=p0)
keep = log['t'].le(cut).to_numpy()
unchanged = bool((reference.loc[keep].to_numpy() == changed.loc[keep].to_numpy()).all())
moved = int((reference.loc[~keep].to_numpy() != changed.loc[~keep].to_numpy()).any(axis=1).sum())
print(f'Perturbation check: {int(later.sum()):,} labels flipped; rows at or before the cut unchanged: {unchanged}; '
      f'later rows that changed: {moved:,}')
assert unchanged
del perturbed, reference, changed
_ = gc.collect()

Brute-force check: 225 rows, 0 mismatches


Perturbation check: 3,968,056 labels flipped; rows at or before the cut unchanged: True; later rows that changed: 3,967,751


### B3. How useful is each feature on validation?

Each validation impression is ranked by one feature at a time. Features where lower values mean more clicks are flipped and marked "lower is better". This is only a screening check. The model decides how to combine the features.

In [10]:
validation_rows = np.flatnonzero(log['partition'].to_numpy() == 1)
lower_is_better = {'art_hours_since_first_exposure': -1, 'user_article_skipped_before': -1, 'user_article_prior_exposures': -1}
screen = []
for mode, frame in history_frames.items():
    scored = frame.iloc[validation_rows].reset_index(drop=True)
    scored['event_key'] = candidate_keys['validation']['event_key'].astype(str).to_numpy()
    scored['label'] = log['label'].to_numpy()[validation_rows]
    table = univariate_table(scored, HISTORY_COLUMNS, sign=lower_is_better)
    table.insert(0, 'mode', mode)
    screen.append(table)
history_screen = pd.concat(screen, ignore_index=True)
diagnostics['history_screen_validation'] = history_screen
wide = history_screen.pivot(index='feature', columns='mode', values=['AUC', 'nDCG@10']).reindex(HISTORY_COLUMNS)
display(wide.round(4))

seen = {mode: float(frame['art_seen_before'].to_numpy()[validation_rows].mean()) for mode, frame in history_frames.items()}
print('Validation exposures on articles already shown before the cutoff:',
      {k: f'{100 * v:.1f}%' for k, v in seen.items()})
print('Validation rows already shown to the same user and skipped (online):',
      f"{100 * history_frames['online']['user_article_skipped_before'].to_numpy()[validation_rows].mean():.1f}%")
skipped = history_frames['online']['user_article_skipped_before'].to_numpy()[validation_rows] == 1
labels = log['label'].to_numpy()[validation_rows]
print(f'Validation CTR when shown and skipped before: {100 * labels[skipped].mean():.2f}% vs not: {100 * labels[~skipped].mean():.2f}%')

AUC                 nDCG@10                
mode                            frozen nightly  online  frozen nightly  online
feature                                                                       
art_prior_exposures             0.4859  0.4896  0.4724  0.2628  0.2704  0.2584
art_prior_clicks                0.5015  0.5127  0.5502  0.2776  0.2872  0.3083
art_prior_ctr                   0.5637  0.5754  0.6464  0.3109  0.3161  0.3669
art_exposures_1h                0.5025  0.5049  0.4682  0.2751  0.2792  0.2623
art_clicks_1h                   0.5130  0.5160  0.5622  0.2872  0.2902  0.3192
art_exposures_6h                0.5062  0.5070  0.4639  0.2776  0.2832  0.2529
art_clicks_6h                   0.5125  0.5244  0.5475  0.2864  0.2983  0.3045
art_exposures_24h               0.4991  0.5023  0.4718  0.2739  0.2789  0.2591
art_clicks_24h                  0.5074  0.5182  0.5516  0.2820  0.2936  0.3079
art_ctr_24h                     0.5600  0.5728  0.6464  0.3108  0.3165  0.3674
art_hours_since_first_exposure  0.5172  0.5178  0.5218  0.2769  0.2783  0.2871
art_seen_before                 0.4868  0.4896  0.4994  0.2606  0.2663  0.2703
subcat_prior_ctr                0.5484  0.5493  0.5620  0.2983  0.2973  0.3023
user_prior_events               0.5000  0.5000  0.5000  0.2705  0.2705  0.2705
user_prior_exposures            0.5000  0.5000  0.5000  0.2705  0.2705  0.2705
user_prior_clicks               0.5000  0.5000  0.5000  0.2705  0.2705  0.2705
user_prior_ctr                  0.5000  0.5000  0.5000  0.2705  0.2705  0.2705
user_article_prior_exposures    0.5031  0.5045  0.5374  0.2712  0.2716  0.2951
user_article_prior_clicks       0.5009  0.5008  0.5004  0.2719  0.2719  0.2711
user_article_skipped_before     0.5034  0.5047  0.5359  0.2713  0.2717  0.2947

Validation exposures on articles already shown before the cutoff: {'online': '99.7%', 'nightly': '75.0%', 'frozen': '49.7%'}
Validation rows already shown to the same user and skipped (online): 15.4%
Validation CTR when shown and skipped before: 2.36% vs not: 4.14%


**Findings**

- Smoothed CTR (`art_prior_ctr`, `art_ctr_24h`) is the strongest single feature, and its strength depends on how up to date the counts are. Validation AUC / nDCG@10: online 0.646 / 0.367, nightly 0.575 / 0.316, frozen 0.564 / 0.311. The share of validation exposures on articles already shown before the cutoff explains the order: 99.7% online, 75.0% nightly, 49.7% frozen. New articles appear every day, so old counts know nothing about many candidates.
- Exposure counts alone rank below random (AUC 0.46 to 0.47). The articles shown most are clicked less per view, so popularity should be measured as clicks per exposure, not views.
- User features score exactly 0.500. They are the same for every candidate in an impression, so on their own they cannot change the order. They can only help together with other features in a model.
- `user_article_skipped_before` is a useful negative signal. 15.4% of validation candidates had already been shown to the same user and skipped, and their CTR is 2.36% against 4.14% for the rest.

### B4. How should the training features be built?

Validation and test features cannot use their own clicks, so their counts are out of date. `history_mode_check_chris.ipynb` trains the same LightGBM ranker with each version of the training features and scores validation, using validation labels only.

| Training file | Validation `nightly` AUC / nDCG@10 | Validation `frozen` AUC / nDCG@10 |
|---|---|---|
| no history features | 0.6226 / 0.3710 | 0.6226 / 0.3710 |
| `online` (up to date) | 0.6235 / 0.3711 | 0.6117 / 0.3642 |
| `nightly` | 0.6357 / 0.3773 | 0.6258 / 0.3728 |
| `mixed` (half up to date, half nightly) | **0.6369 / 0.3800** | **0.6289 / 0.3757** |

- Training on up-to-date counts and scoring out-of-date counts is worse than using no history at all (AUC -0.0109, nDCG@10 -0.0067, both intervals below zero). The model learns to trust counts that are missing at test time.
- Training on out-of-date counts fixes this, and mixing is best. Mixed beats up-to-date training by +0.0134 AUC and +0.0089 nDCG@10 on nightly validation, and by +0.0172 and +0.0115 on frozen validation. It beats nightly training by a small but significant +0.0013 AUC and +0.0028 nDCG@10.
- With up-to-date counts on validation the same model reaches AUC 0.700 and nDCG@10 0.432, so real-time click logging would be worth about +0.05 nDCG@10. This is for the report discussion only.

**What this means.** The group's `train_only` bundle trains on up-to-date counts and scores out-of-date counts, which is the setup that did worse than no history here. `mixed` training only uses training clicks, so it follows the group's rule and is the fix to use if a model includes these features. In the candidate attention model, adding the bundle's CTR features gave no validation gain, which agrees with this check. LightGBM is only a quick check with untuned settings.

In [11]:
def export_history(frame, partition, name):
    out = pd.concat([candidate_keys[partition].reset_index(drop=True), frame.reset_index(drop=True)], axis=1)
    if len(out) != len(candidate_keys[partition]):
        raise ValueError('Row count changed during export')
    write_candidate_table(out, partition, name)
    print(f'{partition}: wrote {name}.csv.gz ({len(out):,} rows)')


for mode, frame in history_frames.items():
    for partition, code_ in zip(PARTITIONS, range(3)):
        rows = np.flatnonzero(log['partition'].to_numpy() == code_)
        export_history(frame.iloc[rows], partition, f'candidate_history_{mode}')

MIX_STALE_SHARE, MIX_SEED = 0.5, 42
train_rows = np.flatnonzero(log['partition'].to_numpy() == 0)
mixed, stale_rows = mix_by_impression(history_frames['online'].iloc[train_rows], history_frames['nightly'].iloc[train_rows],
                                      log['event'].to_numpy()[train_rows], stale_share=MIX_STALE_SHARE, seed=MIX_SEED)
export_history(mixed, 'train', 'candidate_history_mixed')
print(f'Mixed training file: {100 * stale_rows.mean():.1f}% of training rows use nightly (stale) counts')
history_meta = {'smoothing': SMOOTHING, 'window_hours': list(WINDOW_HOURS),
                'global_prior_ctr': round(float(log.loc[log['partition'].eq(0), 'label'].mean()), 6),
                'mixed_training': {'stale_share': MIX_STALE_SHARE, 'seed': MIX_SEED, 'fresh': 'online', 'stale': 'nightly'}}
del history_frames, log
_ = gc.collect()

train: wrote candidate_history_online.csv.gz (3,389,342 rows)


validation: wrote candidate_history_online.csv.gz (2,454,102 rows)


test: wrote candidate_history_online.csv.gz (2,740,998 rows)


train: wrote candidate_history_nightly.csv.gz (3,389,342 rows)


validation: wrote candidate_history_nightly.csv.gz (2,454,102 rows)


test: wrote candidate_history_nightly.csv.gz (2,740,998 rows)


train: wrote candidate_history_frozen.csv.gz (3,389,342 rows)


validation: wrote candidate_history_frozen.csv.gz (2,454,102 rows)


test: wrote candidate_history_frozen.csv.gz (2,740,998 rows)


train: wrote candidate_history_mixed.csv.gz (3,389,342 rows)
Mixed training file: 50.0% of training rows use nightly (stale) counts


## Part C. Entity prominence (insight 4)

Each entity annotation has character offsets and surface forms. The next cell checks that the text at each offset matches the surface form, so the offsets can be used as positions.

| Column | Meaning |
|---|---|
| `has_entity_at_title_start` | An entity starts the title, allowing only leading punctuation (for example `'Trump' visits...`) |
| `title_start_entity_type`, `title_start_type_group` | Type code of that entity, and its group: person (P), organisation (O), place (G), other or none |
| `title_entity_first_position` | Earliest title entity offset divided by title length; 1.0 with `title_entity_absent = 1` when there is none |
| `abstract_entity_first_position` | The same for the abstract |

**Position-weighted entity vector.** Each mention gets the weight `section weight x exp(-decay x position)`, with title weight 1 and abstract weight `a`. An entity in both title and abstract counts once with its larger weight, and the article vector is the weighted mean of its TransE vectors. `a = 1, decay = 0` is the plain average, so the uniform version is one point of the same grid. Mentions without offsets (a few hundred) are given the last position.

In [12]:
ENTITY_COLUMNS = ['news_id', 'title', 'abstract', 'title_entities_parsed', 'abstract_entities_parsed']
alignment = []
for partition in PARTITIONS:
    news = read_news(partition, ENTITY_COLUMNS)
    for section in ('title', 'abstract'):
        matched = total = 0
        for text, entities in zip(news[section], news[f'{section}_entities_parsed']):
            for entity in entities:
                for surface, offset in zip(entity.get('SurfaceForms') or [], entity.get('OccurrenceOffsets') or []):
                    total += 1
                    matched += text[offset:offset + len(surface)] == surface
        alignment.append({'partition': partition, 'section': section, 'occurrences': total,
                          'offset_matches_surface_form_%': round(100 * matched / max(total, 1), 3)})
display(pd.DataFrame(alignment))

,partition,section,occurrences,offset_matches_surface_form_%
0,train,title,51121,100.0
1,train,abstract,92180,100.0
2,validation,title,44812,100.0
3,validation,abstract,79168,100.0
4,test,title,50263,100.0
5,test,abstract,88614,100.0


In [13]:
prominence_by_partition, mentions_by_partition, summary_rows = {}, {}, []
for partition in PARTITIONS:
    news = read_news(partition, ENTITY_COLUMNS)
    mentions = entity_mentions(news)
    features = article_prominence_features(news, mentions)
    mentions_by_partition[partition] = mentions
    prominence_by_partition[partition] = features
    title_first = mentions.loc[mentions['section'].eq('title') & mentions['first_offset'].ge(0)].groupby('news_id')['first_offset'].min()
    summary_rows.append({
        'partition': partition, 'articles': len(features), 'entity_mentions': len(mentions),
        'mentions_without_offsets': int(mentions['first_offset'].lt(0).sum()),
        'title_start_strict_offset_0': int(title_first.eq(0).sum()),
        'title_start_with_leading_punctuation': int(features['has_entity_at_title_start'].sum()),
        'title_entity_absent_%': round(100 * features['title_entity_absent'].mean(), 2),
        'abstract_entity_absent_%': round(100 * features['abstract_entity_absent'].mean(), 2),
        'median_title_first_position': round(float(features.loc[features['title_entity_absent'].eq(0), 'title_entity_first_position'].median()), 3),
    })
    features.to_csv(OUTPUT_DIR / partition / 'news_entity_prominence.csv', index=False, float_format='%.6g')
display(pd.DataFrame(summary_rows))
display(prominence_by_partition['train']['title_start_type_group'].value_counts().rename('train articles'))

,partition,articles,entity_mentions,mentions_without_offsets,title_start_strict_offset_0,title_start_with_leading_punctuation,title_entity_absent_%,abstract_entity_absent_%,median_title_first_position
0,train,43370,130664,488,14253,14364,27.05,27.36,0.083
1,validation,38421,113374,415,12495,12590,27.59,28.21,0.082
2,test,42416,126975,460,13997,14102,26.68,27.59,0.082


title_start_type_group
none            29006
person           4807
organisation     4453
other            2718
place            2386
Name: train articles, dtype: int64

### C2. Are prominent entities linked to clicks?

1. **Pooled training CTR** by lead entity type. This mixes different impressions.
2. **Ranking within each validation impression**, which is what the recommender needs. A feature can show a big pooled CTR gap and still barely change the order inside one impression.

The lead type is also turned into a number using its smoothed training CTR, then scored on validation.

In [14]:
train_rows = read_candidates('train').astype({'news_id': str}).merge(prominence_by_partition['train'], on='news_id', how='left', validate='many_to_one')
pooled = train_rows.groupby('title_start_type_group')['label'].agg(ctr='mean', exposures='size')
pooled['ctr_%'] = (100 * pooled.pop('ctr')).round(2)
display(pooled.sort_values('ctr_%', ascending=False))
group_stats = train_rows.groupby('title_start_type_group')['label'].agg(['sum', 'size'])
prior = train_rows['label'].mean()
type_rate = (group_stats['sum'] + 100 * prior) / (group_stats['size'] + 100)
del train_rows

val_rows = read_candidates('validation').astype({'news_id': str}).merge(prominence_by_partition['validation'], on='news_id', how='left', validate='many_to_one')
val_rows['event_key'] = val_rows['event_key'].astype(str)
val_rows['lead_type_train_ctr'] = val_rows['title_start_type_group'].map(type_rate).fillna(prior)
prominence_screen = univariate_table(val_rows, [
    'has_entity_at_title_start', 'lead_type_train_ctr', 'title_entity_first_position',
    'abstract_entity_first_position', 'title_entity_absent'],
    sign={'title_entity_first_position': -1, 'abstract_entity_first_position': -1, 'title_entity_absent': -1})
diagnostics['prominence_screen_validation'] = prominence_screen
display(prominence_screen)

,exposures,ctr_%
title_start_type_group,,
person,437133,5.99
other,243101,5.61
organisation,281026,5.52
none,2337363,3.60
place,90719,2.26


,feature,direction,impressions,AUC,MRR,nDCG@5,nDCG@10
0,has_entity_at_title_start,higher is better,61894,0.5078,0.2160,0.2209,0.2786
1,lead_type_train_ctr,higher is better,61894,0.5120,0.2184,0.2243,0.2817
2,title_entity_first_position,lower is better,61894,0.5127,0.2163,0.2220,0.2794
3,abstract_entity_first_position,lower is better,61894,0.5016,0.2123,0.2188,0.2751
4,title_entity_absent,lower is better,61894,0.5095,0.2125,0.2164,0.2746


Pooled over training exposures, titles led by a person (5.99% CTR), an organisation (5.52%) or another entity (5.61%) are clicked more than titles with no lead entity (3.60%), and titles led by a place are clicked less (2.26%). Within an impression these features barely change the ranking (AUC 0.508 for the flag, 0.512 for the lead type rate). Most of the pooled gap comes from which impressions contain these articles, not from users preferring them inside an impression. The lead type is more useful than the plain flag, since place goes the other way.

### C3. Testing position-weighted entity vectors

This compares weighting schemes without a model. For each validation impression the user profile is the mean entity vector of the history articles, and each candidate is scored by cosine similarity to it. Only the weighting changes between rows.

- **all**: every validation impression. Candidates or users without an entity vector get score 0.
- **covered**: the same candidate rows for every scheme, where the uniform scheme has both a user profile and a candidate vector.

Two picks are fixed before running the grid, both by covered nDCG@10: the best scheme overall, and the best scheme that keeps abstract entities, since title-only schemes leave many more articles without a vector. Paired 95% intervals check each gain over uniform. Entity vectors come from each partition's own `entity_embedding.vec` (the train file for train and validation, the dev file for test).

In [15]:
SCHEMES = {
    'uniform': (1.0, 0.0),
    'abstract_half': (0.5, 0.0),
    'position_decay_1': (1.0, 1.0),
    'abstract_half_decay_1': (0.5, 1.0),
    'abstract_half_decay_2': (0.5, 2.0),
    'abstract_quarter_decay_2': (0.25, 2.0),
    'abstract_tenth_decay_1': (0.1, 1.0),
    'abstract_tenth_decay_2': (0.1, 2.0),
    'title_only': (0.0, 0.0),
    'title_only_decay_1': (0.0, 1.0),
    'title_only_decay_2': (0.0, 2.0),
}
vector_sources = {'train': 'MINDsmall_train', 'validation': 'MINDsmall_train', 'test': 'MINDsmall_dev'}
entity_tables = {source: load_entity_vectors(RAW_DIR / source / 'entity_embedding.vec') for source in set(vector_sources.values())}
shared = set(entity_tables['MINDsmall_train'][0]) & set(entity_tables['MINDsmall_dev'][0])
gap = max(float(np.abs(entity_tables['MINDsmall_train'][1][entity_tables['MINDsmall_train'][0][e]]
                       - entity_tables['MINDsmall_dev'][1][entity_tables['MINDsmall_dev'][0][e]]).max()) for e in shared)
print(f'Entity vectors: train file {len(entity_tables["MINDsmall_train"][0]):,}, dev file {len(entity_tables["MINDsmall_dev"][0]):,}, '
      f'shared {len(shared):,}, largest difference on shared entities {gap}')

events = read_events('validation', history=True)
candidates = read_candidates('validation')
news_ids = prominence_by_partition['validation']['news_id'].astype(str).tolist()
row_of = pd.Series(np.arange(len(news_ids)), index=news_ids)
event_of = pd.Series(np.arange(len(events)), index=events['event_key'].astype(str))
history = events[['event_key', 'history_ids']].explode('history_ids').dropna()
history_article = row_of.reindex(history['history_ids'].astype(str)).to_numpy()
history_event = event_of.reindex(history['event_key'].astype(str)).to_numpy()
candidate_article = row_of.reindex(candidates['news_id'].astype(str)).to_numpy()
candidate_event = event_of.reindex(candidates['event_key'].astype(str)).to_numpy()
index, matrix = entity_tables['MINDsmall_train']


def profile_scores(vectors, has_vector):
    profile = np.zeros((len(events), vectors.shape[1]), dtype='float32')
    counts = np.zeros(len(events))
    usable = has_vector[history_article]
    np.add.at(profile, history_event[usable], vectors[history_article[usable]])
    np.add.at(counts, history_event[usable], 1)
    profile /= np.maximum(counts, 1)[:, None]
    profile_norm = np.linalg.norm(profile, axis=1)
    vector_norm = np.linalg.norm(vectors, axis=1)
    scores = np.zeros(len(candidates))
    for start in range(0, len(candidates), 500_000):
        e = candidate_event[start:start + 500_000]
        a = candidate_article[start:start + 500_000]
        scores[start:start + 500_000] = (profile[e] * vectors[a]).sum(1) / np.maximum(profile_norm[e] * vector_norm[a], 1e-12)
    valid = (counts[candidate_event] > 0) & has_vector[candidate_article]
    return np.where(valid, scores, 0.0), valid


scored = pd.DataFrame({'event_key': candidates['event_key'].astype(str).to_numpy(), 'label': candidates['label'].to_numpy()})
rows, covered = [], None
for name, (abstract_weight, decay) in SCHEMES.items():
    weights = mention_weights(mentions_by_partition['validation'], abstract_weight=abstract_weight, decay=decay)
    vectors, missing = aggregate_entity_vectors(news_ids, mentions_by_partition['validation'], weights, index, matrix)
    scored[name], valid = profile_scores(vectors, missing == 0)
    if covered is None:
        covered = valid
    rows.append({'scheme': name, 'abstract_weight': abstract_weight, 'decay': decay,
                 'articles_with_vector_%': round(100 * float((missing == 0).mean()), 2),
                 **{f'all_{k}': v for k, v in impression_metrics(scored, name).items() if k != 'impressions'},
                 **{f'covered_{k}': v for k, v in impression_metrics(scored.loc[covered], name).items()}})
weighting = pd.DataFrame(rows)
diagnostics['entity_weighting_validation'] = weighting
display(weighting)

Entity vectors: train file 26,904, dev file 22,893, shared 18,346, largest difference on shared entities 0.0


,scheme,abstract_weight,decay,articles_with_vector_%,all_AUC,all_MRR,all_nDCG@5,all_nDCG@10,covered_impressions,covered_AUC,covered_MRR,covered_nDCG@5,covered_nDCG@10
0,uniform,1.00,0.0,85.41,0.5438,0.2338,0.2411,0.3013,51022,0.5519,0.2490,0.2589,0.3238
1,abstract_half,0.50,0.0,85.41,0.5457,0.2355,0.2435,0.3032,51022,0.5553,0.2512,0.2621,0.3264
2,position_decay_1,1.00,1.0,85.41,0.5464,0.2365,0.2444,0.3045,51022,0.5564,0.2525,0.2631,0.3280
3,abstract_half_decay_1,0.50,1.0,85.41,0.5469,0.2365,0.2448,0.3047,51022,0.5576,0.2528,0.2639,0.3283
4,abstract_half_decay_2,0.50,2.0,85.41,0.5476,0.2372,0.2460,0.3055,51022,0.5583,0.2535,0.2652,0.3293
5,abstract_quarter_decay_2,0.25,2.0,85.41,0.5482,0.2376,0.2466,0.3062,51022,0.5595,0.2541,0.2661,0.3303
6,abstract_tenth_decay_1,0.10,1.0,85.41,0.5497,0.2386,0.2483,0.3073,51022,0.5620,0.2554,0.2681,0.3316
7,abstract_tenth_decay_2,0.10,2.0,85.41,0.5493,0.2383,0.2479,0.3072,51022,0.5613,0.2551,0.2678,0.3315
8,title_only,0.00,0.0,71.85,0.5422,0.2414,0.2532,0.3101,51022,0.5531,0.2593,0.2747,0.3354
9,title_only_decay_1,0.00,1.0,71.85,0.5419,0.2415,0.2536,0.3101,51022,0.5525,0.2596,0.2752,0.3355


In [16]:
ranked = weighting.sort_values('covered_nDCG@10', ascending=False)
SELECTED_SCHEME = ranked.iloc[0]['scheme']
SELECTED_FULL_COVERAGE = ranked.loc[ranked['abstract_weight'].gt(0)].iloc[0]['scheme']
for label, name in [('Best overall', SELECTED_SCHEME), ('Best keeping abstract entities', SELECTED_FULL_COVERAGE)]:
    print(f'{label}: {name} (abstract weight {SCHEMES[name][0]}, decay {SCHEMES[name][1]})')
paired = pd.DataFrame([{'comparison': f'{name} minus uniform', **paired_difference(scored.loc[covered], 'uniform', name, metric)}
                       for name in dict.fromkeys([SELECTED_SCHEME, SELECTED_FULL_COVERAGE, 'abstract_half_decay_1'])
                       for metric in ('AUC', 'nDCG@10') if name != 'uniform'])
diagnostics['entity_weighting_paired'] = paired
display(paired)
del events, candidates, history, scored, val_rows
_ = gc.collect()

Best overall: title_only_decay_1 (abstract weight 0.0, decay 1.0)
Best keeping abstract entities: abstract_tenth_decay_1 (abstract weight 0.1, decay 1.0)


,comparison,metric,mean_difference,ci_low,ci_high,impressions
0,title_only_decay_1 minus uniform,AUC,0.0006,-0.0022,0.0034,51022
1,title_only_decay_1 minus uniform,nDCG@10,0.0117,0.0095,0.0138,51022
2,abstract_tenth_decay_1 minus uniform,AUC,0.0100,0.0086,0.0115,51022
3,abstract_tenth_decay_1 minus uniform,nDCG@10,0.0077,0.0063,0.0091,51022
4,abstract_half_decay_1 minus uniform,AUC,0.0057,0.0048,0.0066,51022
5,abstract_half_decay_1 minus uniform,nDCG@10,0.0045,0.0035,0.0055,51022


**Results.** The effects are small because a cosine profile is a weak ranker on its own (about 0.55 AUC).

- Giving abstract entities and later positions less weight helps on both metrics. The first guess (abstract weight 0.5, decay 1) beats uniform by +0.0057 AUC [0.0048, 0.0066] and +0.0045 nDCG@10 [0.0035, 0.0055]. The best scheme that keeps abstract entities, `abstract_tenth_decay_1`, gains +0.0100 AUC [0.0086, 0.0115] and +0.0077 nDCG@10 [0.0063, 0.0091], with the same 85% article coverage as uniform.
- Position decay helps when abstract entities are included (covered nDCG@10 0.3238 for uniform, 0.3280 for `position_decay_1`) but adds almost nothing for titles alone, which are short.
- Title-only schemes give the best nDCG@10 (+0.0117) but no significant AUC gain, and 28% of articles lose their entity vector (14% for the other schemes).

**Recommendation.** Use `abstract_tenth_decay_1` as the entity vector, keep `uniform` for comparison and `title_only_decay_1` as a variant, and keep the title-start and first-position columns as article inputs. A learned encoder can reweight entities itself, so the choice should be checked inside a model.

**Follow-up.** The shared bundle stores `abstract_tenth_decay_1` as its `prominence` vector and keeps `uniform` as the default. In the candidate attention model the two gave about the same validation nDCG@10 (0.3937 with prominence, 0.3942 with uniform).

In [17]:
exported = []
for partition in PARTITIONS:
    index, matrix = entity_tables[vector_sources[partition]]
    news_ids = prominence_by_partition[partition]['news_id'].astype(str).tolist()
    flags = pd.DataFrame({'news_id': news_ids})
    for name in dict.fromkeys(['uniform', SELECTED_FULL_COVERAGE, SELECTED_SCHEME]):
        abstract_weight, decay = SCHEMES[name]
        weights = mention_weights(mentions_by_partition[partition], abstract_weight=abstract_weight, decay=decay)
        vectors, missing = aggregate_entity_vectors(news_ids, mentions_by_partition[partition], weights, index, matrix)
        np.save(OUTPUT_DIR / partition / f'news_entity_vectors_{name}.npy', vectors)
        flags[f'entity_vector_missing_{name}'] = missing
        exported.append({'partition': partition, 'scheme': name, 'rows': vectors.shape[0], 'dim': vectors.shape[1],
                         'missing_%': round(100 * float(missing.mean()), 2)})
    flags.to_csv(OUTPUT_DIR / partition / 'news_entity_vector_index.csv', index=False)
display(pd.DataFrame(exported))

,partition,scheme,rows,dim,missing_%
0,train,uniform,43370,100,13.83
1,train,abstract_tenth_decay_1,43370,100,13.83
2,train,title_only_decay_1,43370,100,27.59
3,validation,uniform,38421,100,14.59
4,validation,abstract_tenth_decay_1,38421,100,14.59
5,validation,title_only_decay_1,38421,100,28.15
6,test,uniform,42416,100,13.89
7,test,abstract_tenth_decay_1,42416,100,13.89
8,test,title_only_decay_1,42416,100,27.23


## Export summary

Each partition folder contains the files below. Join candidate-level files on `(event_key, candidate_position)` and article-level files on `news_id`. Row *i* of each `.npy` matrix matches row *i* of `news_entity_vector_index.csv`.

In [18]:
for partition in PARTITIONS:
    encoded_by_partition[partition].to_csv(OUTPUT_DIR / partition / 'news_category_encoding.csv', index=False)
vocab['category'].to_csv(OUTPUT_DIR / 'category_vocab.csv', index=False)
vocab['subcategory'].to_csv(OUTPUT_DIR / 'subcategory_vocab.csv', index=False)
vocab['pairs'].to_csv(OUTPUT_DIR / 'subcategory_pairs.csv', index=False)
for name, table in diagnostics.items():
    table.to_csv(OUTPUT_DIR / f'diagnostic_{name}.csv', index=False)

dictionary = pd.DataFrame([
    ('category_id', 'news_category_encoding', 'News metadata', 'Training-sorted category code, 0 unknown', 'int32', 'Article', 'No'),
    ('subcategory_id', 'news_category_encoding', 'News metadata', '(category, subcategory) code; rare/unseen pairs use the category __RARE__ bucket; 0 unknown', 'int32', 'Article', 'No'),
    ('category_unknown, subcategory_backoff', 'news_category_encoding', 'News metadata', 'Flags for unknown category and non-individual subcategory code', 'int8', 'Article', 'No'),
    ('candidate_history_subcategory_share', 'candidate_subcategory_match', 'Behaviors + News', 'Share of supplied history in the candidate (category, subcategory) pair', 'float32', 'Candidate', 'No'),
    ('candidate_history_subcategory_count, history_subcategory_missing', 'candidate_subcategory_match', 'Behaviors + News', 'Matching history count; flag for no usable history subcategory', 'int', 'Candidate', 'No'),
    *[(column, 'candidate_history_{online,nightly,frozen}', 'Behaviors (labels of earlier impressions only)', 'See Part B table', 'float32' if 'ctr' in column or 'hours' in column else 'int', 'Candidate', 'Earlier impressions only')
      for column in HISTORY_COLUMNS],
    ('has_entity_at_title_start', 'news_entity_prominence', 'news.tsv entity offsets', 'Entity starts the title (leading punctuation allowed)', 'int8', 'Article', 'No'),
    ('title_start_entity_type, title_start_type_group', 'news_entity_prominence', 'news.tsv entity types', 'Type of the lead entity; person/organisation/place/other/none', 'string', 'Article', 'No'),
    ('title_entity_first_position, abstract_entity_first_position', 'news_entity_prominence', 'news.tsv entity offsets', 'Earliest offset / text length; 1.0 when absent', 'float32', 'Article', 'No'),
    ('title_entity_absent, abstract_entity_absent', 'news_entity_prominence', 'news.tsv entity offsets', 'No located entity in that section', 'int8', 'Article', 'No'),
    ('news_entity_vectors_<scheme>.npy', 'entity vectors', 'news.tsv + entity_embedding.vec', 'Weighted mean of 100-d TransE vectors; zero row when missing', 'float32[100]', 'Article', 'No'),
], columns=['feature', 'file', 'source', 'construction', 'data_type', 'level', 'uses_labels'])
dictionary.to_csv(OUTPUT_DIR / 'feature_dictionary.csv', index=False)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()


provenance = {
    'source_dataset_version': manifest['dataset_version'],
    'source_manifest_sha256': sha256(PROCESSED_DIR / 'manifest.json'),
    'partitions': PARTITIONS,
    'category_min_articles': MIN_ARTICLES,
    'history': history_meta,
    'history_modes': {'online': 'time < t from any partition',
                      'nightly': 'time < midnight starting the impression day, every partition',
                      'frozen': 'train: time < t; later partitions: time < partition start'},
    'entity_weight_formula': 'section_weight * exp(-decay * normalised_position); entity counted once with max weight',
    'entity_schemes_exported': list(dict.fromkeys(['uniform', SELECTED_FULL_COVERAGE, SELECTED_SCHEME])),
    'entity_scheme_selected_overall': SELECTED_SCHEME,
    'entity_scheme_selected_keeping_abstract': SELECTED_FULL_COVERAGE,
    'entity_scheme_selection_rule': 'highest covered nDCG@10 on validation (overall, and among abstract_weight > 0), fixed before the grid',
    'entity_scheme_parameters': {name: dict(zip(['abstract_weight', 'decay'], SCHEMES[name]))
                                 for name in dict.fromkeys(['uniform', SELECTED_FULL_COVERAGE, SELECTED_SCHEME])},
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(provenance, indent=2) + '\n')

written = sorted(p for p in OUTPUT_DIR.rglob('*') if p.is_file())
listing = pd.DataFrame({'file': [str(p.relative_to(OUTPUT_DIR)) for p in written],
                        'MB': [round(p.stat().st_size / 1e6, 1) for p in written]})
display(listing)
print(f'Total {listing["MB"].sum():.0f} MB written in {(time.time() - started) / 60:.1f} minutes')
display(dictionary)

,file,MB
0,category_vocab.csv,0.0
1,diagnostic_entity_weighting_paired.csv,0.0
2,diagnostic_entity_weighting_validation.csv,0.0
3,diagnostic_history_screen_validation.csv,0.0
4,diagnostic_prominence_screen_validation.csv,0.0
5,diagnostic_subcategory_match_validation.csv,0.0
6,feature_dictionary.csv,0.0
7,manifest.json,0.0
8,subcategory_pairs.csv,0.0
9,subcategory_vocab.csv,0.0


Total 738 MB written in 16.8 minutes


,feature,file,source,construction,data_type,level,uses_labels
0,category_id,news_category_encoding,News metadata,"Training-sorted category code, 0 unknown",int32,Article,No
1,subcategory_id,news_category_encoding,News metadata,"(category, subcategory) code; rare/unseen pair...",int32,Article,No
2,"category_unknown, subcategory_backoff",news_category_encoding,News metadata,Flags for unknown category and non-individual ...,int8,Article,No
3,candidate_history_subcategory_share,candidate_subcategory_match,Behaviors + News,Share of supplied history in the candidate (ca...,float32,Candidate,No
4,"candidate_history_subcategory_count, history_s...",candidate_subcategory_match,Behaviors + News,Matching history count; flag for no usable his...,int,Candidate,No
5,art_prior_exposures,"candidate_history_{online,nightly,frozen}",Behaviors (labels of earlier impressions only),See Part B table,int,Candidate,Earlier impressions only
6,art_prior_clicks,"candidate_history_{online,nightly,frozen}",Behaviors (labels of earlier impressions only),See Part B table,int,Candidate,Earlier impressions only
7,art_prior_ctr,"candidate_history_{online,nightly,frozen}",Behaviors (labels of earlier impressions only),See Part B table,float32,Candidate,Earlier impressions only
8,art_exposures_1h,"candidate_history_{online,nightly,frozen}",Behaviors (labels of earlier impressions only),See Part B table,int,Candidate,Earlier impressions only
9,art_clicks_1h,"candidate_history_{online,nightly,frozen}",Behaviors (labels of earlier impressions only),See Part B table,int,Candidate,Earlier impressions only


## How the models use these features

The models do not read these files directly. `03_Unified_Feature_Engineering.ipynb` rebuilds the features in the shared bundle (`outputs/features/unified_v1`) with the same subcategory codes, the `train_only` history mode, and `abstract_tenth_decay_1` as the `prominence` entity vector. This notebook is where those features come from and the evidence for them.